# AMOH · Sentinel-1 SAR and maritime context

Optional Earth Engine development notebook for the Gerlache Strait area.
Remote data is not required by the local AMOH MVP. Authenticate outside the repository with `earthengine authenticate` before running Earth Engine cells.

In [ ]:
from typing import Any

# Earth Engine and geemap expose dynamic client-side objects.
import ee as _ee  # pyright: ignore[reportMissingTypeStubs]
import geemap as _geemap  # pyright: ignore[reportMissingTypeStubs, reportUnusedImport]

ee: Any = _ee
geemap: Any = _geemap
ee.Initialize()
print("Earth Engine initialized")

In [ ]:
CENTER = (-64.8, -62.5)
AOI = ee.Geometry.Rectangle([-65.0, -66.0, -60.0, -63.5])

sar = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(AOI)
    .filterDate("2024-01-01", "2024-04-01")
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "HH"))
    .select("HH")
)
sar_median = sar.median().clip(AOI)
print("Sentinel-1 scenes:", sar.size().getInfo())

In [ ]:
visualization: dict[str, object] = {
    "min": -25,
    "max": 5,
    "palette": ["071c2c", "14566a", "83d8d0", "f2c500"],
}
m = geemap.Map(center=CENTER, zoom=7)
m.addLayer(sar_median, visualization, "Sentinel-1 HH median - Q1 2024")
m.addLayer(ee.Feature(AOI), {"color": "f2c500"}, "AMOH AOI")
m

## Maritime context layers
The following collections are optional context and must retain their source, license and acquisition metadata. They do not prove vessel presence or environmental impact.

In [ ]:
# Global Fishing Watch native Earth Engine asset requested for AMOH.
gfw_vessels = ee.FeatureCollection("GLOBAL_FISHING_WATCH/V1/vessels")
gfw_aoi = gfw_vessels.filterBounds(AOI)
print("GFW features in AOI:", gfw_aoi.size().getInfo())

In [ ]:
# Local/open-source overlays are intentionally loaded from explicit paths.
# They are not silently treated as GEE assets.
QUANTARCTICA_ROUTES = '../data/external/quantarctica/iaato_routes.gpkg'
CMEMS_ICE = '../data/external/cmems/sea_ice_concentration.nc'
EMODNET_AIS = '../data/external/emodnet/ais_positions.parquet'
print('Configure and verify licenses before loading:', QUANTARCTICA_ROUTES, CMEMS_ICE, EMODNET_AIS)

## Interpretation limits
- Sentinel-1 scenes are snapshots, not continuous real time.
- AIS absence is not vessel absence.
- The observer GPS position is not automatically the target vessel position.
- A visual overlay is context; it is not an environmental impact score.